In [1]:
import pandas as pd
df = pd.read_csv(r'C:\Users\Dell\OneDrive\Desktop\HR_Attrition.csv')
df_raw = df.copy()
print(df.shape)

(1470, 35)


In [2]:
df =df.drop(columns=['EmployeeCount','Over18','StandardHours','EmployeeNumber'])
print(df.shape)

(1470, 31)


In [3]:
cat_col= df.select_dtypes(include='object').columns
print(cat_col)

Index(['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender',
       'JobRole', 'MaritalStatus', 'OverTime'],
      dtype='object')


In [4]:
df['Attrition']= df['Attrition'].map({'Yes':1,'No':0})
df['Gender']= df['Gender'].map({'Male':1,'Female':0})
df['OverTime']= df['OverTime'].map({'Yes':1,'No':0})
print(df[['Attrition','Gender','OverTime']])
                                      

      Attrition  Gender  OverTime
0             1       0         1
1             0       1         0
2             1       1         1
3             0       0         1
4             0       1         0
...         ...     ...       ...
1465          0       1         0
1466          0       1         0
1467          0       1         1
1468          0       1         0
1469          0       1         0

[1470 rows x 3 columns]


In [5]:
df = pd.get_dummies(df,columns=['BusinessTravel','Department','EducationField','JobRole','MaritalStatus'],drop_first=True)
print(df.shape)

(1470, 45)


In [6]:
print(df.head())

   Age  Attrition  DailyRate  DistanceFromHome  Education  \
0   41          1       1102                 1          2   
1   49          0        279                 8          1   
2   37          1       1373                 2          2   
3   33          0       1392                 3          4   
4   27          0        591                 2          1   

   EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  JobLevel  ...  \
0                        2       0          94               3         2  ...   
1                        3       1          61               2         2  ...   
2                        4       1          92               2         1  ...   
3                        4       0          56               3         1  ...   
4                        1       1          40               3         1  ...   

   JobRole_Human Resources  JobRole_Laboratory Technician  JobRole_Manager  \
0                    False                          False           

In [7]:
from sklearn.model_selection import train_test_split
X= df.drop(columns=['Attrition'])
y=df['Attrition']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape,X_test.shape)

(1176, 44) (294, 44)


In [16]:
from sklearn.linear_model import LogisticRegression

model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

print("Model Trained Successfully")

Model Trained Successfully


C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

model=LogisticRegression(max_iter=1000)
model.fit(X_train_scaled,y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [25]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix

y_pred= model.predict(X_test_scaled)

print("Accuracy:",accuracy_score(y_test,y_pred))
print("Precison:",precision_score(y_test,y_pred))
print("Recall:",recall_score(y_test,y_pred))
print("F1 Score:",f1_score(y_test,y_pred))
print("Confusion matrix:",confusion_matrix(y_test,y_pred))


Accuracy: 0.8775510204081632
Precison: 0.5483870967741935
Recall: 0.4358974358974359
F1 Score: 0.4857142857142857
Confusion matrix: [[241  14]
 [ 22  17]]


In [10]:
import pandas as pd
coefficients=pd.DataFrame({
        'Feature':X.columns,
        'Coefficient':model.coef_[0]
})

coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients=coefficients.sort_values(by='Abs_Coefficient',ascending=False)

print(coefficients.head(10))

                             Feature  Coefficient  Abs_Coefficient
13                          OverTime     0.975324         0.975324
35     JobRole_Laboratory Technician     0.780808         0.780808
25  BusinessTravel_Travel_Frequently     0.717950         0.717950
21                    YearsAtCompany     0.668289         0.668289
22                YearsInCurrentRole    -0.641188         0.641188
43              MaritalStatus_Single     0.625524         0.625524
12                NumCompaniesWorked     0.509736         0.509736
41      JobRole_Sales Representative     0.506542         0.506542
23           YearsSinceLastPromotion     0.500860         0.500860
24              YearsWithCurrManager    -0.462789         0.462789


In [12]:
coefficients.to_csv('feature_importance.csv',index = True)